In [3]:
#r "C:\Users\user\Desktop\Kristina\practice2026\task17\bin\Debug\net10.0\task17.dll"
#r "nuget:ScottPlot,5.0.54"

using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Threading;
using ScottPlot;
using task17;

public class HeavyCommand : ICommand
{
    private int _rem;
    private readonly IScheduler _scheduler;
    public bool IsCompleted => _rem <= 0;
    
    public HeavyCommand(int n, IScheduler scheduler = null)
    {
        _rem = n;
        _scheduler = scheduler;
    }

    public void Execute()
    {
        if (_rem <= 0) return;
        
        _rem--;
        double x = 0;
        for (int i = 0; i < 100000; i++)
            x += Math.Sin(i) * Math.Sin(i);
        
        if (_rem > 0 && _scheduler != null)
            _scheduler.Add(this);
    }
}

var counts = new int[] { 2, 4, 6, 8, 10 };
var seqTimes = new double[counts.Length];
var rrTimes = new double[counts.Length];

Console.WriteLine("Замеры производительности...\n");

for (int i = 0; i < counts.Length; i++)
{
    int n = counts[i];

    double seqTotal = 0;
    for (int run = 0; run < 3; run++)
    {
        var sw = Stopwatch.StartNew();
        for (int j = 0; j < n; j++)
        {
            var cmd = new HeavyCommand(3);
            while (!cmd.IsCompleted) cmd.Execute();
        }
        sw.Stop();
        seqTotal += sw.Elapsed.TotalMilliseconds;
    }
    seqTimes[i] = seqTotal / 3;

    double rrTotal = 0;
    for (int run = 0; run < 3; run++)
    {
        var s = new Scheduler();
        var st = new ServerThread(s);
        var cmds = new HeavyCommand[n];
        for (int j = 0; j < n; j++)
        {
            cmds[j] = new HeavyCommand(3, s);
            s.Add(cmds[j]);
        }

        var sw = Stopwatch.StartNew();
        st.Start();
        while (!cmds.All(c => c.IsCompleted)) Thread.Sleep(1);
        sw.Stop();
        st.HardStop();
        st.Join();
        rrTotal += sw.Elapsed.TotalMilliseconds;
    }
    rrTimes[i] = rrTotal / 3;
    
    Console.WriteLine($"Команд: {n,2} : Round Robin: {rrTimes[i],8:F1} мс : Без планировщика: {seqTimes[i],8:F1} мс ");
}

var plot = new Plot();
plot.XLabel("Количество команд");
plot.YLabel("Время (мс)");

var xs = counts.Select(x => (double)x).ToArray();
var s1 = plot.Add.Scatter(xs, seqTimes);
s1.LegendText = "Без планировщика";
s1.LineWidth = 2;
s1.MarkerSize = 8;

var s2 = plot.Add.Scatter(xs, rrTimes);
s2.LegendText = "Round Robin";
s2.LineWidth = 2;
s2.MarkerSize = 8;

plot.ShowLegend();
plot.SavePng("res.png", 800, 600);
display(HTML($"<img src='res.png' width='700'/>"));

var report = new System.Text.StringBuilder();
report.AppendLine("ОТЧЁТ");
report.AppendLine($"Дата: {DateTime.Now}");
report.AppendLine();
report.AppendLine("Команд | Round Robin | Без планировщика");

for (int i = 0; i < counts.Length; i++)
    report.AppendLine($"{counts[i],6} | {seqTimes[i],14:F1} мс | {rrTimes[i],9:F1} мс");

File.WriteAllText("report.txt", report.ToString());
Console.WriteLine("\nОтчёт сохранён в report.txt");

Installed Packages ScottPlot, 5.0.54

Замеры производительности...

Команд:  2 : Round Robin:     16.4 мс : Без планировщика:     22.2 мс 
Команд:  4 : Round Robin:     32.9 мс : Без планировщика:     26.2 мс 
Команд:  6 : Round Robin:     52.5 мс : Без планировщика:     48.0 мс 
Команд:  8 : Round Robin:     65.8 мс : Без планировщика:     55.0 мс 
Команд: 10 : Round Robin:     74.0 мс : Без планировщика:     66.0 мс 



Отчёт сохранён в report.txt
